### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [17]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [18]:
### Read all the pdfs inside the directory
def process_all_pdfs(pdf_directory):
    """Processes all PDF files in the specified directory and returns a list of documents."""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Successfully processed {pdf_file}, extracted {len(documents)} documents.")

        except Exception as e:
            print(f"Error processing file {pdf_file}: {e}")
    return all_documents

# Process all PDFs in the specified directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process.
Processing file: ..\data\pdf\kaushik blood report.pdf
Successfully processed ..\data\pdf\kaushik blood report.pdf, extracted 2 documents.


In [19]:
all_pdf_documents

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-08-22T14:35:53+00:00', 'source': '..\\data\\pdf\\kaushik blood report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'kaushik blood report.pdf', 'file_type': 'pdf'}, page_content='DIAGNOSTIC REPORT\nMAKAN NO. 0189 ANDHIYARI BAGH, SOUTH THANA\nTIWARIPUR,GORAKHPUR\nPATIENT NAME :  ADITYA KAUSHIK\nPATIENT ID       : ADITM406605810\nACCESSION NO : 0185YE006091 AGE/SEX    :20 Years Male\nABHA NO          :\nDRAWN      :25/05/2025 10:46:36\nRECEIVED  :25/05/2025 12:33:12\nREPORTED  :25/05/2025 14:42:41\nREF. DOCTOR :  SELF\nCLIENT PATIENT ID:\nCODE/NAME & ADDRESS  : C000141270\nANDHIYARI BAGH COLLECTION CENTRE\nGORAKHPUR 273001\n8081533776\nFinal Results Biological Reference IntervalTest Report Status Units\nBIOCHEMISTRY\nBLOOD GLUCOSE RANDOM RWA\nGLUCOSE RANDOM, PLASMA 78.8 70 - 140 mg/dL\nSGOT & SGPT RWA\nASPARTATE AMINOTRANSFERASE, SERUM\nASPARTATE AMINOTRANSFERASE\n(

In [20]:
### Text Splitter get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Splits documents into smaller chunks using RecursiveCharacterTextSplitter."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # show example of the first chunk
    if split_docs:
        print(f"Example of the first chunk:\n{split_docs[0].page_content[:200]}...")
        print(f"Metadata of the first chunk:\n{split_docs[0].metadata}")

    return split_docs

In [21]:
chunks = split_documents(all_pdf_documents)
chunks

Split 2 documents into 6 chunks.
Example of the first chunk:
DIAGNOSTIC REPORT
MAKAN NO. 0189 ANDHIYARI BAGH, SOUTH THANA
TIWARIPUR,GORAKHPUR
PATIENT NAME :  ADITYA KAUSHIK
PATIENT ID       : ADITM406605810
ACCESSION NO : 0185YE006091 AGE/SEX    :20 Years Male
...
Metadata of the first chunk:
{'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-08-22T14:35:53+00:00', 'source': '..\\data\\pdf\\kaushik blood report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'kaushik blood report.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-08-22T14:35:53+00:00', 'source': '..\\data\\pdf\\kaushik blood report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'kaushik blood report.pdf', 'file_type': 'pdf'}, page_content='DIAGNOSTIC REPORT\nMAKAN NO. 0189 ANDHIYARI BAGH, SOUTH THANA\nTIWARIPUR,GORAKHPUR\nPATIENT NAME :  ADITYA KAUSHIK\nPATIENT ID       : ADITM406605810\nACCESSION NO : 0185YE006091 AGE/SEX    :20 Years Male\nABHA NO          :\nDRAWN      :25/05/2025 10:46:36\nRECEIVED  :25/05/2025 12:33:12\nREPORTED  :25/05/2025 14:42:41\nREF. DOCTOR :  SELF\nCLIENT PATIENT ID:\nCODE/NAME & ADDRESS  : C000141270\nANDHIYARI BAGH COLLECTION CENTRE\nGORAKHPUR 273001\n8081533776\nFinal Results Biological Reference IntervalTest Report Status Units\nBIOCHEMISTRY\nBLOOD GLUCOSE RANDOM RWA\nGLUCOSE RANDOM, PLASMA 78.8 70 - 140 mg/dL\nSGOT & SGPT RWA\nASPARTATE AMINOTRANSFERASE, SERUM\nASPARTATE AMINOTRANSFERASE\n(

### Embedding and VectorStoreDB

In [22]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\ML Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializes the embedding manager.
        
        Args:
            model_name (str): The name of the sentence transformer model to use for generating embeddings.

        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully loaded embedding model: {self.model_name} with embedding dimension {self.model.get_sentence_embedding_dimension()}.")
        except Exception as e:
            print(f"Error loading embedding model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of strings to generate embeddings for.

        Returns:
            np.ndarray: An array of embeddings for the input texts.
        """

        if not self.model:
            raise ValueError("Embedding model is not loaded. Please check the model name and try again.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Finished generating embeddings: {embeddings.shape}")
        return embeddings

## intialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2183.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully loaded embedding model: all-MiniLM-L6-v2 with embedding dimension 384.


### VectorStore

In [38]:
class VectorStore:
    """A simple vector store implementation using ChromaDB to store and retrieve document embeddings."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/chromadb_data"):
        """Initializes the vector store.
        
        Args:
            collection_name (str): The name of the ChromaDB collection to use for storing embeddings.
            persist_directory (str): The directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            # create persist directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "PDF document embeddings for RAG"})

            print(f"Successfully initialized ChromaDB collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Adds documents and their corresponding embeddings to the vector store.
        
        Args:
            documents (List[Any]): A list of document objects to add to the store. Each document should have a unique identifier in its metadata.
            embeddings (np.ndarray): An array of embeddings corresponding to the input documents.
        """
        if not self.collection:
            raise ValueError("Vector store collection is not initialized. Please check the initialization and try again.")

        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")

        # Prepare data for insertion
        ids = []  # Generate unique IDs for each document
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  # Generate a unique ID for the document
            ids.append(doc_id)

            # Create a copy of the document's metadata and add additional fields
            metadata = doc.metadata.copy()
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for JSON serialization

        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Successfully initialized ChromaDB collection: pdf_documents
Existing documents in collection: 0


In [39]:
chunks

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-08-22T14:35:53+00:00', 'source': '..\\data\\pdf\\kaushik blood report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'kaushik blood report.pdf', 'file_type': 'pdf'}, page_content='DIAGNOSTIC REPORT\nMAKAN NO. 0189 ANDHIYARI BAGH, SOUTH THANA\nTIWARIPUR,GORAKHPUR\nPATIENT NAME :  ADITYA KAUSHIK\nPATIENT ID       : ADITM406605810\nACCESSION NO : 0185YE006091 AGE/SEX    :20 Years Male\nABHA NO          :\nDRAWN      :25/05/2025 10:46:36\nRECEIVED  :25/05/2025 12:33:12\nREPORTED  :25/05/2025 14:42:41\nREF. DOCTOR :  SELF\nCLIENT PATIENT ID:\nCODE/NAME & ADDRESS  : C000141270\nANDHIYARI BAGH COLLECTION CENTRE\nGORAKHPUR 273001\n8081533776\nFinal Results Biological Reference IntervalTest Report Status Units\nBIOCHEMISTRY\nBLOOD GLUCOSE RANDOM RWA\nGLUCOSE RANDOM, PLASMA 78.8 70 - 140 mg/dL\nSGOT & SGPT RWA\nASPARTATE AMINOTRANSFERASE, SERUM\nASPARTATE AMINOTRANSFERASE\n(

In [40]:
### convert the chunks to text and generate embeddings
texts = [doc.page_content for doc in chunks]

## generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 6 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.48it/s]

Finished generating embeddings: (6, 384)
Successfully added 6 documents to the vector store.
Total documents in collection after addition: 6
